In [28]:
from ollama import ollama_generate

response = ollama_generate(
    model='qwen3:14b',
#     prompt="""[{'role': 'user', 'content':'What is the weather in Toronto?'}] ,
    
#     Available tools=[{
#       'type': 'function',
#       'function': {
#         'name': 'get_current_weather',
#         'description': 'Get the current weather for a city',
#         'parameters': {
#           'type': 'object',
#           'properties': {
#             'city': {
#               'type': 'string',
#               'description': 'The name of the city',
#             },
#           },
#           'required': ['city'],
#         },
#       },
#     }]"""
# )  
    # prompt = """ 
    # What is the weather in Toronto?
    # """

    prompt = """
    System Instructions : You are a helpful AI assistant that can check weather information. You have access to a weather tool.

AVAILABLE TOOLS:
[{
  'type': 'function',
  'function': {
    'name': 'get_current_weather',
    'description': 'Get the current weather for a city',
    'parameters': {
      'type': 'object',
      'properties': {
        'city': {
          'type': 'string',
          'description': 'The name of the city',
        },
      },
      'required': ['city'],
    },
  },
}]

FUNCTION CALLING RULES:
1. When users ask about weather, use the get_current_weather function
2. Use this EXACT JSON format for function calls:

```json
{{
    "function": "get_current_weather",
    "parameters": {{
        "city": "city_name"
    }}
}}
```

3. Always validate that the city parameter is provided
4. Provide helpful weather information using the available tool

Be precise and helpful when providing weather information.

User : What is the weather in Toronto?

Assistant : """)

print(response)
response = response['response']

{'model': 'qwen3:14b', 'created_at': '2025-10-09T09:25:30.091272754Z', 'response': '<think>\nOkay, the user is asking about the weather in Toronto. I need to use the get_current_weather function. Let me check the rules again. The function requires the city parameter, which is provided here as Toronto. So I should generate the correct JSON call. Make sure the city name is spelled correctly. Then, once I get the data from the tool, I can relay the information back to the user. But for now, just the function call is needed.\n</think>\n\n```json\n{\n    "function": "get_current_weather",\n    "parameters": {\n        "city": "Toronto"\n    }\n}\n```', 'done': True, 'done_reason': 'stop', 'context': [151644, 872, 271, 262, 739, 38297, 549, 1446, 525, 264, 10950, 15235, 17847, 429, 646, 1779, 9104, 1995, 13, 1446, 614, 2615, 311, 264, 9104, 5392, 382, 73078, 5146, 40263, 510, 58, 515, 220, 364, 1313, 1210, 364, 1688, 751, 220, 364, 1688, 1210, 341, 262, 364, 606, 1210, 364, 455, 11080, 69364

In [30]:
print(response)

<think>
Okay, the user is asking about the weather in Toronto. I need to use the get_current_weather function. Let me check the rules again. The function requires the city parameter, which is provided here as Toronto. So I should generate the correct JSON call. Make sure the city name is spelled correctly. Then, once I get the data from the tool, I can relay the information back to the user. But for now, just the function call is needed.
</think>

```json
{
    "function": "get_current_weather",
    "parameters": {
        "city": "Toronto"
    }
}
```


In [31]:
import json
def parse_function_call(response):
    """
    Parse and validate function calls from AI response.
    
    Args:
        response (str): AI response containing potential function calls
        
    Returns:
        tuple: (success: bool, function_call: dict, error: str)
    """
    try:
        if "```json" in response:
            # Extract JSON from markdown code block
            json_start = response.find("```json") + 7
            json_end = response.find("```", json_start)
            json_str = response[json_start:json_end].strip()
            
            function_call = json.loads(json_str)
            function_name = function_call.get("function")
            parameters = function_call.get("parameters", {})
            
            # Validate function call
            # is_valid, missing_params, error_msg = tool_registry.validate_tool_call(
            #     function_name, parameters
            # )
            
            return True, function_call, None
            
                
        return False, {}, "No function call found in response"
        
    except json.JSONDecodeError as e:
        return False, {}, f"Invalid JSON format: {str(e)}"
    except Exception as e:
        return False, {}, f"Parsing error: {str(e)}"

print(response)
_,function_call,_ = parse_function_call(response)
print(function_call)

<think>
Okay, the user is asking about the weather in Toronto. I need to use the get_current_weather function. Let me check the rules again. The function requires the city parameter, which is provided here as Toronto. So I should generate the correct JSON call. Make sure the city name is spelled correctly. Then, once I get the data from the tool, I can relay the information back to the user. But for now, just the function call is needed.
</think>

```json
{
    "function": "get_current_weather",
    "parameters": {
        "city": "Toronto"
    }
}
```
{'function': 'get_current_weather', 'parameters': {'city': 'Toronto'}}


In [ ]:
def format_tool_call_for_openai(function_call):
    """
    Format the tool call response to match OpenAI's expected format.
    
    Args:
        function_call (dict): The function call dictionary to format.
        
    Returns:
        dict: Formatted function call for OpenAI.
    """
    if not function_call:
        return {}
    
    formatted_call = {
        "tool_name": function_call.get("function", ""),
        "arguments": function_call.get("parameters", {})
    }
    return formatted_call


# Example usage
formatted_call = format_tool_call_for_openai(function_call)
print(formatted_call)
